# AI Travel Planner — Advanced GenAI Version ✈️🌍
An extensible travel-planning assistant that turns destination, dates, budget, interests, and travel style into a structured itinerary. Demonstrates optional LLM itinerary generation, JSON output, budget allocation, preference matching, validation, and exports.

> **Important:** This is a portfolio starter, not a live booking system. Prices, opening hours, visa rules, weather, and transport schedules are not verified. Confirm details with official providers before traveling.

## Features
- Reusable trip-request model and input validation
- Optional OpenAI-compatible LLM itinerary generation
- Deterministic fallback when no API key is configured
- Budget breakdown and basic budget checks
- Activity-to-interest matching
- Itinerary structure and schedule validation
- JSON and Markdown exports

Workflow: `Trip Request → LLM/Fallback Planner → Validation → Budget Summary → Export`

## 1. Install dependencies
Run this cell in Jupyter.

In [ ]:
%pip install -q pandas python-dotenv openai

## 2. Configuration

In [ ]:
import os, json, re
from dataclasses import dataclass, asdict
from datetime import date, timedelta
from pathlib import Path
from typing import Any
import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_KEY = os.getenv("OPENAI_API_KEY", "")
BASE_URL = os.getenv("OPENAI_BASE_URL", "").strip() or None
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
print("API key configured:", bool(API_KEY), "| Model:", MODEL)

## 3. Define a trip request

In [ ]:
@dataclass
class TripRequest:
    destination: str
    start_date: str
    end_date: str
    budget: float
    currency: str = "INR"
    travelers: int = 1
    interests: tuple[str, ...] = ("culture", "food", "nature")
    travel_style: str = "balanced"
    accommodation_style: str = "mid-range"
    transport_preference: str = "public transport"
    dietary_preferences: tuple[str, ...] = ()
    accessibility_needs: tuple[str, ...] = ()
    notes: str = ""

    def validate(self):
        start, end = date.fromisoformat(self.start_date), date.fromisoformat(self.end_date)
        if not self.destination.strip(): raise ValueError("Destination cannot be empty.")
        if end < start: raise ValueError("end_date must be on or after start_date.")
        if self.budget <= 0: raise ValueError("Budget must be greater than zero.")
        if self.travelers < 1: raise ValueError("Travelers must be at least 1.")
        if self.travel_style not in {"relaxed", "balanced", "packed"}:
            raise ValueError("travel_style must be relaxed, balanced, or packed.")

    @property
    def days(self):
        return (date.fromisoformat(self.end_date) - date.fromisoformat(self.start_date)).days + 1

trip = TripRequest(
    destination="Jaipur, Rajasthan, India",
    start_date="2027-01-10", end_date="2027-01-13",
    budget=25000, currency="INR", travelers=2,
    interests=("history", "architecture", "local food", "markets"),
    travel_style="balanced", accommodation_style="mid-range",
    transport_preference="cab and walking",
    dietary_preferences=("vegetarian options preferred",),
    notes="Mix major sights with relaxed evenings."
)
trip.validate()
print(f"{trip.destination} | {trip.days} days | {trip.travelers} travelers | {trip.currency} {trip.budget:,.0f}")

## 4. Build a structured LLM prompt

In [ ]:
SYSTEM_PROMPT = '''
You are a careful travel-planning assistant. Create realistic, preference-aware itineraries.
Do not claim you checked live prices, current opening hours, availability, weather, visa rules, or transport schedules.
Mark uncertain details for verification and respect budget, dates, group size, dietary and accessibility needs.
Return only valid JSON with this shape:
{
 "trip_summary":"string",
 "assumptions":["string"],
 "days":[{"date":"YYYY-MM-DD","theme":"string","activities":[
   {"time":"Morning|Afternoon|Evening","name":"string","category":"string",
    "description":"string","estimated_cost_per_person":0,"duration_hours":1.0,
    "area":"string","verification_needed":true}],
   "food_notes":["string"],"practical_notes":["string"]}],
 "budget_breakdown":{"accommodation":0,"food":0,"local_transport":0,"activities":0,"buffer":0},
 "safety_and_verification":["string"]
}
Use numeric estimates in the requested currency. Keep daily schedules practical.
'''

def build_user_prompt(trip):
    payload = asdict(trip)
    for key in ("interests", "dietary_preferences", "accessibility_needs"):
        payload[key] = list(payload[key])
    payload["number_of_days"] = trip.days
    return "Plan a trip from this request. Return only JSON.\n" + json.dumps(payload, ensure_ascii=False, indent=2)

print(build_user_prompt(trip))

## 5. Optional LLM generation

In [ ]:
def extract_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return json.loads(text)

def generate_with_llm(trip):
    if not API_KEY:
        raise RuntimeError("OPENAI_API_KEY is not configured.")
    from openai import OpenAI
    kwargs = {"api_key": API_KEY}
    if BASE_URL: kwargs["base_url"] = BASE_URL
    client = OpenAI(**kwargs)
    response = client.chat.completions.create(
        model=MODEL, temperature=0.3,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(trip)}
        ]
    )
    return extract_json(response.choices[0].message.content or "{}")

try:
    plan = generate_with_llm(trip)
    plan_source = f"LLM provider ({MODEL})"
except Exception as exc:
    print("LLM generation unavailable:", exc)
    print("Using deterministic demonstration fallback.")
    plan, plan_source = None, "fallback demo"

## 6. Fallback planner
This is a generic mock itinerary, not a verified destination guide.

In [ ]:
def fallback_plan(trip):
    start = date.fromisoformat(trip.start_date)
    templates = [
        ("Arrival and orientation", [
            ("Morning","Arrive and settle in","other","Check in, rest, and confirm local transport.",0,1.0),
            ("Afternoon","Explore a central neighborhood","culture","Walk at a relaxed pace and note useful services.",300,2.0),
            ("Evening","Try a local dinner spot","food","Choose a venue and verify current hours.",700,1.5)]),
        ("Landmarks and history", [
            ("Morning","Visit a heritage attraction","history","Check official opening hours and ticket rules.",600,2.5),
            ("Afternoon","Explore nearby architecture","culture","Group nearby sights to reduce travel time.",300,2.0),
            ("Evening","Visit a market or café","shopping","Confirm operating days before visiting.",400,1.5)]),
        ("Food, craft, and culture", [
            ("Morning","Try a cultural or craft experience","culture","Check provider details and booking conditions.",800,2.0),
            ("Afternoon","Explore local food","food","Check ingredients against dietary needs.",900,2.0),
            ("Evening","Free time or viewpoint","nature","Choose a safe location and verify conditions.",200,1.5)]),
        ("Flexible final day", [
            ("Morning","Choose a favorite attraction","culture","Use this slot for a verified attraction or rest.",500,2.0),
            ("Afternoon","Souvenir shopping and departure prep","shopping","Allow time for traffic and checkout.",500,1.5)])
    ]
    days = []
    for i in range(trip.days):
        theme, template = templates[min(i, len(templates)-1)]
        activities = [{
            "time": t, "name": n, "category": c, "description": desc,
            "estimated_cost_per_person": cost, "duration_hours": duration,
            "area": "Area to be confirmed", "verification_needed": True
        } for t,n,c,desc,cost,duration in template]
        days.append({
            "date": (start + timedelta(days=i)).isoformat(), "theme": theme,
            "activities": activities,
            "food_notes": ["Verify current venue details and dietary suitability."],
            "practical_notes": ["Confirm travel times and opening hours."]
        })
    budget = {
        "accommodation": round(trip.budget * .35),
        "food": round(trip.budget * .22),
        "local_transport": round(trip.budget * .15),
        "activities": round(trip.budget * .18),
        "buffer": round(trip.budget * .10)
    }
    return {
        "trip_summary": f"Demonstration {trip.days}-day plan for {trip.destination}.",
        "assumptions": [
            "Fallback is generic, not destination-verified.",
            "Budget is a starting estimate, not a quote.",
            "Live prices, hours, and availability have not been checked."
        ],
        "days": days, "budget_breakdown": budget,
        "safety_and_verification": [
            "Confirm official opening hours and ticket rules.",
            "Check transport options and local advisories.",
            "Keep emergency contacts and booking details available."
        ]
    }

if plan is None:
    plan = fallback_plan(trip)
print("Plan source:", plan_source, "| Days:", len(plan.get("days", [])))

## 7. View the itinerary

In [ ]:
def itinerary_table(plan):
    rows = []
    for day in plan.get("days", []):
        for activity in day.get("activities", []):
            rows.append({
                "date": day.get("date"), "day_theme": day.get("theme"),
                "time": activity.get("time"), "activity": activity.get("name"),
                "category": activity.get("category"),
                "duration_hours": activity.get("duration_hours"),
                "estimated_cost_per_person": activity.get("estimated_cost_per_person"),
                "verification_needed": activity.get("verification_needed", True)
            })
    return pd.DataFrame(rows)

itinerary_df = itinerary_table(plan)
display(itinerary_df)

## 8. Validate dates, schedule, and budget

In [ ]:
def validate_plan(plan, trip):
    issues = []
    days = plan.get("days")
    if not isinstance(days, list):
        return ["'days' must be a list."]
    expected = {(date.fromisoformat(trip.start_date) + timedelta(days=i)).isoformat()
                for i in range(trip.days)}
    actual = {d.get("date") for d in days if isinstance(d, dict)}
    if expected - actual: issues.append(f"Missing dates: {sorted(expected-actual)}")
    if actual - expected: issues.append(f"Unexpected dates: {sorted(actual-expected)}")
    for idx, day in enumerate(days, start=1):
        activities = day.get("activities", []) if isinstance(day, dict) else []
        if not activities: issues.append(f"Day {idx} has no activities.")
        hours = sum(float(a.get("duration_hours") or 0) for a in activities)
        if hours > 10: issues.append(f"Day {idx} schedules {hours:.1f} activity hours; review workload.")
        for activity in activities:
            cost = activity.get("estimated_cost_per_person", 0)
            if not isinstance(cost, (int, float)) or cost < 0:
                issues.append(f"Invalid cost on day {idx}: {activity.get('name')}")
    breakdown = plan.get("budget_breakdown", {})
    if isinstance(breakdown, dict):
        total = sum(v for v in breakdown.values() if isinstance(v, (int, float)))
        if total > trip.budget:
            issues.append(f"Budget allocation ({total:,.0f}) exceeds budget ({trip.budget:,.0f}).")
    return issues or ["No basic structural issues detected; factual accuracy is not verified."]

validation_issues = validate_plan(plan, trip)
for item in validation_issues:
    print("•", item)

## 9. Budget summary

In [ ]:
budget_df = pd.DataFrame([
    {"category": k, "estimated_amount": v, "currency": trip.currency}
    for k, v in plan.get("budget_breakdown", {}).items()
    if isinstance(v, (int, float))
])
display(budget_df)
allocated = budget_df["estimated_amount"].sum() if not budget_df.empty else 0
print(f"Budget: {trip.currency} {trip.budget:,.0f}")
print(f"Allocated: {trip.currency} {allocated:,.0f}")
print(f"Remaining / (over budget): {trip.currency} {trip.budget-allocated:,.0f}")

## 10. Rank activities against interests

In [ ]:
def interest_match_score(category, interests):
    category = (category or "").lower()
    terms = [x.lower() for x in interests]
    if any(term in category or category in term for term in terms):
        return 2
    aliases = {
        "history": {"heritage", "museum", "monument", "architecture"},
        "food": {"restaurant", "cafe", "market", "cuisine"},
        "nature": {"park", "garden", "viewpoint", "outdoors"},
        "shopping": {"craft", "souvenir", "bazaar"},
        "culture": {"craft", "performance", "local"}
    }
    return int(any(term in aliases and any(word in category for word in aliases[term]) for term in terms))

if not itinerary_df.empty:
    itinerary_df["interest_match_score"] = itinerary_df["category"].apply(
        lambda c: interest_match_score(str(c), trip.interests))
    display(itinerary_df.sort_values(["interest_match_score", "date"], ascending=[False, True]))

## 11. Export JSON and Markdown

In [ ]:
OUTPUT_DIR = Path("travel_planner_output")
OUTPUT_DIR.mkdir(exist_ok=True)
json_path = OUTPUT_DIR / "travel_plan.json"
json_path.write_text(json.dumps({
    "request": asdict(trip), "plan_source": plan_source,
    "plan": plan, "validation_issues": validation_issues
}, ensure_ascii=False, indent=2), encoding="utf-8")

def plan_to_markdown(trip, plan, source):
    lines = [
        f"# Travel Plan: {trip.destination}", "",
        f"- **Dates:** {trip.start_date} to {trip.end_date} ({trip.days} days)",
        f"- **Travelers:** {trip.travelers}",
        f"- **Budget:** {trip.currency} {trip.budget:,.0f}",
        f"- **Plan source:** {source}", "",
        "## Summary", plan.get("trip_summary", ""), "", "## Itinerary"
    ]
    for day in plan.get("days", []):
        lines += ["", f"### {day.get('date')} — {day.get('theme','')}"]
        for a in day.get("activities", []):
            lines.append(
                f"- **{a.get('time','Time TBD')}: {a.get('name','Activity')}** "
                f"({a.get('category','other')}; {trip.currency} "
                f"{a.get('estimated_cost_per_person',0)} per person) — "
                f"{a.get('description','')} [Verify: {a.get('verification_needed',True)}]"
            )
        for note in day.get("food_notes", []): lines.append(f"- Food note: {note}")
        for note in day.get("practical_notes", []): lines.append(f"- Practical note: {note}")
    lines += ["", "## Budget Breakdown"]
    lines += [f"- {k.replace('_',' ').title()}: {trip.currency} {v:,.0f}"
              for k,v in plan.get("budget_breakdown", {}).items()]
    lines += ["", "## Safety and Verification"]
    lines += [f"- {x}" for x in plan.get("safety_and_verification", [])]
    lines += ["", "> Estimates and itinerary details are not live-verified. Confirm information before booking."]
    return "\n".join(lines)

md_path = OUTPUT_DIR / "travel_plan.md"
md_path.write_text(plan_to_markdown(trip, plan, plan_source), encoding="utf-8")
print("Saved:", json_path.resolve())
print("Saved:", md_path.resolve())

## 12. Future improvements
- Add licensed maps/places search and cite source links.
- Integrate weather and time-zone APIs with retrieval timestamps.
- Add RAG over trusted destination guides and official tourism sources.
- Build a Streamlit or React interface with an editable itinerary.
- Add saved trips, database persistence, tests, retries, rate limits, and prompt-injection defenses.
- Keep API keys in environment variables; never commit `.env`.